In [1]:
# %pip install sentence_transformers
# %pip install celery
# %pip install langchain_ollama

In [2]:
from dotenv import load_dotenv
load_dotenv()

True

In [3]:
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer(
    "BAAI/bge-small-en-v1.5"
)

E:\Hackathon\WellNest\WellNest\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|█████████████████████████████████████████████████████████████████████████| 199/199 [00:00<00:00, 2055.38it/s]


In [4]:
import os
MONGO_URL = os.getenv("MONGO_URL")

In [17]:
from motor.motor_asyncio import AsyncIOMotorClient

client = AsyncIOMotorClient(MONGO_URL)

db = client["medical_ai"]
chats = db["chats"]
session_summaries = db["session_summaries"]
patient_profiles = db["patient_profiles"]

In [18]:
from pymongo import MongoClient

client = MongoClient(MONGO_URL)
db_sync = client["medical_ai"]
chats_sync = db_sync["chats"]
session_summaries_sync = db_sync["session_summaries"]
patient_profiles_sync = db_sync["patient_profiles"]

In [ ]:
{
  "fields": [
    {
      "type": "vector",
      "path": "embedding",
      "numDimensions": 384,
      "similarity": "cosine"
    },
    {
      "type": "filter",
      "path": "user_id"
    }
  ]
}

In [7]:
from datetime import datetime

async def append_message(
    session_id: str,
    user_id: str,
    message: dict
):
    await chats.update_one(
        {
            "session_id": session_id,
            "user_id": user_id
        },
        {
            "$push": {
                "messages": message
            },
            "$set": {
                "last_activity": datetime.utcnow(),
                "status": "active",
                "summary_generated": False
            },
            "$inc": {
                "total_messages": 1
            },
            "$setOnInsert": {
                "session_id": session_id,
                "user_id": user_id,
                "created_at": datetime.utcnow()
            }
        },
        upsert=True
    )

In [8]:
# celery_app.py
from celery import Celery
import os

REDIS_URL = os.getenv("REDIS_URL")

celery_app = Celery(
    "medical",
    broker=REDIS_URL,
    backend=None   # IMPORTANT
)

celery_app.conf.timezone = "UTC"

In [9]:
celery_app.conf.beat_schedule = {
    "scan-inactive-sessions": {
        "task": "tasks.find_inactive_sessions",
        "schedule": 300.0
    }
}

In [10]:
from datetime import datetime, timedelta

@celery_app.task
def find_inactive_sessions():

    cutoff = datetime.utcnow() - timedelta(minutes=30)

    sessions = chats_sync.find({
        "last_activity": {"$lt": cutoff},
        "summary_generated": {"$ne": True},
        "status": {"$nin": ["summarizing", "summarized"]},
        "$expr": {
            "$gt": [
                "$total_messages",
                {
                    "$ifNull": ["$last_summarized_total_messages", 0]
                }
            ]
        }
    })

    for session in sessions:

        # atomic lock to avoid duplicate workers
        result = chats_sync.update_one(
            {
                "session_id": session["session_id"],
                "user_id":session["user_id"]
            },
            {
                "$set": {
                    "status": "summarizing",
                    "last_summarized_total_messages": session["total_messages"]
                }
            }
        )

        if result.modified_count == 1:
            summarize_session.delay(session["session_id"])

In [11]:
from datetime import datetime

async def mark_inactive_and_summarize(session_id: str):
    """
    Marks a session as inactive and triggers Celery summarization.
    """

    # 1. Update session state in MongoDB
    result = await chats.update_one(
        {
            "session_id": session_id,
            "user_id": user_id
        },
        {
            "$set": {
                "status": "inactive",
                "last_activity": datetime.utcnow()
            }
        }
    )

    # 2. Only trigger summarization if session exists
    if result.matched_count == 0:
        return {"status": "session_not_found"}

    # 3. Trigger Celery task
    summarize_session.delay(session_id)

    return {
        "status": "marked_inactive_and_summarization_triggered",
        "session_id": session_id
    }

In [12]:
from langchain_ollama import ChatOllama

llm = ChatOllama(
    model="qwen3:4b",
    temperature=0
)

E:\Hackathon\WellNest\WellNest\.venv\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.5.0) or chardet (7.4.3)/charset_normalizer (3.4.3) doesn't match a supported version!
  warnings.warn(


In [13]:
PROMPT = """
You are an expert medical conversation summarizer.

Your task is to analyze the complete conversation between the patient and the AI assistant.

Your responsibilities are:

1. Generate a concise summary of the conversation.

2. Extract all information discussed during THIS session:
   - symptoms
   - medications
   - medical conditions
   - allergies

3. Determine whether this conversation reveals NEW or UPDATED long-term patient information that should become part of the patient's persistent medical profile.

Only include information in profile_updates if it represents long-term medical history such as:
- Chronic diseases
- Previous surgeries
- Permanent allergies
- Long-term medications
- Family history
- Lifestyle factors (smoking, alcohol, etc.)

Do NOT include temporary illnesses (cold, fever, stomach ache, etc.) unless they become confirmed chronic conditions.

For every profile item, generate a short medical description.

Return ONLY valid JSON.

{
    "summary": "...",

    "structured_data": {
        "symptoms": [],
        "medications": [],
        "conditions": [],
        "allergies": []
    },

    "profile_updates": {

        "conditions": [
            {
                "name": "",
                "description": ""
            }
        ],

        "surgeries": [
            {
                "name": "",
                "description": ""
            }
        ],

        "allergies": [
            {
                "name": "",
                "description": ""
            }
        ],

        "current_medications": [
            {
                "name": "",
                "description": ""
            }
        ],

        "family_history": [
            {
                "name": "",
                "description": ""
            }
        ],

        "lifestyle": [
            {
                "name": "",
                "description": ""
            }
        ]
    }
}

Conversation:

{chat}
"""

In [15]:
from datetime import datetime

def update_patient_profile(user_id: str, updates: dict, session_id: str):
    profile = patient_profiles_sync.find_one({"user_id": user_id})

    if profile is None:
        profile = {
            "user_id": user_id,
            "conditions": [],
            "surgeries": [],
            "allergies": [],
            "current_medications": [],
            "family_history": [],
            "lifestyle": [],
            "updated_at": datetime.utcnow()
        }

    for category in [
        "conditions",
        "surgeries",
        "allergies",
        "current_medications",
        "family_history",
        "lifestyle"
    ]:

        existing = {
            item["name"].lower(): item
            for item in profile.get(category, [])
        }

        for new_item in updates.get(category, []):

            key = new_item["name"].strip().lower()

            if key in existing:

                existing[key]["description"] = new_item["description"]

                existing[key]["last_confirmed"] = datetime.utcnow()

                existing[key]["mention_count"] = (
                    existing[key].get("mention_count", 1) + 1
                )

                sessions = existing[key].setdefault("source_sessions", [])

                if session_id not in sessions:
                    sessions.append(session_id)

            else:

                new_item["first_seen"] = datetime.utcnow()

                new_item["last_confirmed"] = datetime.utcnow()

                new_item["mention_count"] = 1

                new_item["source_sessions"] = [session_id]

                profile[category].append(new_item)

    profile["updated_at"] = datetime.utcnow()

    patient_profiles_sync.replace_one(
        {"user_id": user_id},
        profile,
        upsert=True
    )

In [16]:
@celery_app.task
def summarize_session(session_id: str,user_id:str):
    session = chats.find_one({"session_id": session_id})

    # 🔥 guard conditions
    if session["status"] != "inactive":
        return  # user came back → cancel summarization

    if session.get("summary_generated"):
        return

    text = "\n".join(
        f"{m['role']}: {m['content']}"
        for m in session["messages"]
    )

    result = llm.invoke(PROMPT.format(chat=text))
    data = json.loads(result.content)

    embedding = embedder.encode(
                                data["summary"],
                                normalize_embeddings=True
                            ).tolist()

    session_summaries_sync.update_one(
        {
            "session_id": session_id,
             "user_id":user_id
        },
        {
            "$set": {
                "summary": data["summary"],
                "structured_data": {
                    "symptoms": data["symptoms"],
                    "medications": data["medications"],
                    "conditions": data["conditions"],
                    "allergies": data["allergies"]
                },
                "embedding": embedding
            }
        },
        upsert=True
    )

    update_patient_profile(
        user_id=user_id,
        updates=data["profile_updates"],
        session_id=session_id
    )
    
    chats_sync.update_one(
            {
                "session_id": session["session_id"],
            },
            {
                "$set": {
                    "status": "summarized",
                    "last_summarized_total_messages": session["total_messages"]
                }
            }
        )